<a href="https://colab.research.google.com/github/navya1870/NLP-Semantic-Analyser/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

#Collecting Data

In [ ]:
#ab jo file h hmare pass vo text file h aur usme sbhi me jo text h vo emotion se semicolon se separate kr rkha h
#so csv = comma separated values if we provide the separator we can make again a  csv file
df = pd.read_csv('train.txt',sep = ';',header = None ,names = ['text','emotion'])

In [ ]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


#Text Cleaning

In [ ]:
df.isna().sum() #checking null values
#got no null value

,0
text,0
emotion,0


In [ ]:
df['emotion'].unique()
#now we need to convert these objects into numbers

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [ ]:
unique_emotions = df['emotion'].unique()
emotions_numbers = {}
i = 0
for emo in unique_emotions:
  emotions_numbers[emo] = i
  i +=1
df['emotion'] = df['emotion'].map(emotions_numbers)

In [ ]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [ ]:
#lowercasing
df['text'] = df['text'].apply(lambda x: x.lower())

In [ ]:
#remove punctuations
#Method 1
import string
def remove_punctuations(text):
  for punc in string.punctuation:
    text = text.replace(punc,'')
  return text
df['text'] = df['text'].apply(remove_punctuations)

In [ ]:
#Method 2
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))
df['text'] = df['text'].apply(remove_punc)

In [ ]:
#remove numbers
def remove_numbers(txt):
  new = ''
  for i in txt:
    if not i.isdigit():
      new = new +i
    return new
    df['text'] = df['text'].apply(remove_numbers)

In [ ]:
#remove emojis
#hrr ek letter ke pass apni ek ascii value hoti h ab chahe vo lowercase ho ya uppercase everyone has it
#similarly there is one more thing called unicode, now that unicode is for everything like for numbers emoji sb kuch ke liye hota h
#first 127 values of ascii and unique code is same cause both are for letters
def remove_emojis(txt):
  new = ''
  for i in txt:
    if i.isascii():
      new = new+i
  return new
  df['text'] = df['text'].apply(remove_emojis)

In [ ]:
#remove stopwords
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt')
#punkt is kind of a library jo words ko tokenize krne me help krti h
nltk.download('stopwords')
#stopwords to download krne hi pdte h cuz inhe hi to remove krna h

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
#in our data there is english only so we used english stopwords only
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [ ]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [ ]:
#we are gonna tokenize each text and then it would be converted into a  list then we will compare each text list with stopwords
#if the given text list contain the stopwords as a word it would be removed from the list
def remove(txt):
  words = txt.split()
  #here i have used text.split() but agr mera punkt kaam kr jata h jo idhr unfortunately kr ni rha to fir i would have used words_tokenize(txt)
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)
df['text'] = df['text'].apply(remove)

In [ ]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

#Bag Of Words

In [ ]:
#using bag of words
from sklearn.feature_extraction.text import CountVectorizer
vectorizer= CountVectorizer()
x = vectorizer.fit_transform(df['text'])
print('Vocabulary:',vectorizer.get_feature_names_out())
print('Bag of words matrix:\n',x.toarray())

Vocabulary: ['aa' 'aaaaaaand' 'aaaaand' ... 'zum' 'zumba' 'zz']
Bag of words matrix:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [ ]:
len(vectorizer.get_feature_names_out()) #size of the vocabulary

15046

isme use krne se to ye gnda sa print hora cuz vese bhi ye bag of words ghatiya hota h so new data pr kr rhe h ye

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(ngram_range=(2,2))
documents = ["I love pizza",'Pizza is the best','I love pasta','Pasta is great']
x = vectorizer.fit_transform(documents)
print('Vocabulary:',vectorizer.get_feature_names_out())
print('Bag of words matrix:\n',x.toarray())

Vocabulary: ['is great' 'is the' 'love pasta' 'love pizza' 'pasta is' 'pizza is'
 'the best']
Bag of words matrix:
 [[0 0 0 1 0 0 0]
 [0 1 0 0 0 1 1]
 [0 0 1 0 0 0 0]
 [1 0 0 0 1 0 0]]


is count vectorizer me already sare text preprocessing ke features h like lowercase stopwords remove krna sb kuch so we can directly also use this but sometime for instance if we want to keep some of the stopwords and remove rest then we should do the text processing on our own

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
documents = ["I love pizza",'Pizza is the best','I love pasta','Pasta is great']
x = vectorizer.fit_transform(documents)
print('Vocabulary:',vectorizer.get_feature_names_out())
print('Bag of words matrix:\n',x.toarray())

Vocabulary: ['best' 'great' 'is' 'love' 'pasta' 'pizza' 'the']
Bag of words matrix:
 [[0.         0.         0.         0.70710678 0.         0.70710678
  0.        ]
 [0.55528266 0.         0.43779123 0.         0.         0.43779123
  0.55528266]
 [0.         0.         0.         0.70710678 0.70710678 0.
  0.        ]
 [0.         0.66767854 0.52640543 0.         0.52640543 0.
  0.        ]]


here values khali 0 and 1 hi nhi h bich ki values bhi h which give a better

In [ ]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [ ]:
from sklearn.model_selection import train_test_split
x = df['text']
y = df['emotion']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

#implying TF-IDF and bag of words

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
bow = CountVectorizer()
x_train_bow = bow.fit_transform(x_train)
x_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116059 stored elements and shape (12800, 13361)>

In [ ]:
x_test_bow = bow.transform(x_test)

we will use naive bayes and logistic regression model as they both are the best models for machine leaning NLP.. we can also use SVM model that is also pretty good for it

for text classification multinomial naive bayes is used and gaussian naive bayes is used for normal data

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
nb_model = MultinomialNB()
nb_model.fit(x_train_bow,y_train)

MultinomialNB()

In [ ]:
pred_nb = nb_model.predict(x_test_bow)

In [ ]:
accuracy = accuracy_score(y_test,pred_nb)
accuracy

0.768125

#accuracy of bag of words is 76.81 percent

In [ ]:
tfidf = TfidfVectorizer()
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

In [ ]:
nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf,y_train)

MultinomialNB()

In [ ]:
pred_nb2 = nb2_model.predict(x_test_tfidf)

In [ ]:
accuracy = accuracy_score(y_test,pred_nb2)
accuracy

0.6609375

#isme to aur bhi km ho gyi accuracy..accuracy by tfidf using NB is 66 percent

#using logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 1000)
model.fit(x_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [ ]:
log_pred = model.predict(x_test_tfidf)

In [ ]:
accuracy = accuracy_score(y_test,log_pred)
accuracy

0.8628125

#sbse bdiya isme aayi h accuracy which is 86 percent that is achieved by using logistic model using TF IDF

In [ ]:
from sklearn.linear_model import LogisticRegression
model_bow_log= LogisticRegression(max_iter = 1000)
model_bow_log.fit(x_train_bow,y_train)

LogisticRegression(max_iter=1000)

In [ ]:
pred_bow_log = model_bow_log.predict(x_test_bow)

In [ ]:
accuracy = accuracy_score(y_test,pred_bow_log)
accuracy

0.8896875

isme aur bhi bdiya accuracy aa gyi that is of 88 percent that is achieved by using logistic regression model and bag of words

#using SVM

In [ ]:
from sklearn.svm import SVC
svm_model = SVC()
svm_model.fit(x_train_tfidf,y_train)

SVC()

In [ ]:
pred_svm = svm_model.predict(x_test_tfidf)

In [ ]:
accuracy = accuracy_score(y_test,pred_svm)
accuracy

0.85125

In [ ]:
svm_model_bow = SVC()
svm_model_bow.fit(x_train_bow,y_train)

SVC()

In [ ]:
pred_svm_bow = svm_model_bow.predict(x_test_bow)
accuracy = accuracy_score(y_test,pred_svm_bow)
accuracy

0.8225

#so even using svm we got accuracy above 80 but highest accuracy is received from logistic regression using bag of words so we will use this model to make our project

ab is model se hi ui bnaege yayyyyy!!!


In [ ]:
import joblib
joblib.dump(model_bow_log,"model.pkl")
joblib.dump(bow,"vectorizer.pkl")

['vectorizer.pkl']

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1
